# SEC 正式申报日期边界修复

保留首次受理时间审计，针对 AMD 2011-01-21 唯一的受理日早于正式申报日事件，按两者较晚日期之后首个 Qlib 交易日校正。该事件原本为定性未知。

In [1]:
import os, json, hashlib, datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_acceptance_clock_audit_20261023'
plan=json.loads((card/'filing_date_repair_plan.json').read_text());sha=lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(card/'result.json')==plan['original_result_sha256']
source=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023/quarter_event_adjudication_result.json'
assert sha(source)==plan['quarter_event_result_sha256']
original=json.loads((card/'result.json').read_text());events=json.loads(source.read_text())['selected_events']
assert len(original['rows'])==len(events)==126
by_accession={x['accession']:x for x in events}
assert len(by_accession)==126
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-01',end_time='2026-10-09',freq='day')).normalize()
assert calendar.is_monotonic_increasing and calendar.is_unique
changes=[];unchanged=0
for row in original['rows']:
    event=by_accession[row['accession']]
    header_day=datetime.datetime.strptime(row['SGML_acceptance_header'],'%Y%m%d%H%M%S').date()
    filing_day=datetime.date.fromisoformat(event['filing_date'])
    later=max(header_day,filing_day)
    future=calendar[calendar.date>later]
    assert len(future)>0
    corrected=str(future[0].date())
    if corrected!=row['next_Qlib_session']:
        changes.append({'issuer':row['issuer'],'accession':row['accession'],'raw_sha256':row['raw_sha256'],'accepted_header':row['SGML_acceptance_header'],'SEC_filing_date':event['filing_date'],'prior_acceptance_based_Qlib_session':row['next_Qlib_session'],'corrected_first_Qlib_session_after_later_date':corrected,'source_value':event['revenue_outlook']})
    else:unchanged+=1
assert len(changes)==1 and unchanged==125
exc=plan['expected_exception'];change=changes[0]
assert all(change[a]==exc[b] for a,b in [('issuer','issuer'),('accession','accession'),('accepted_header','accepted_header'),('SEC_filing_date','SEC_filing_date'),('prior_acceptance_based_Qlib_session','original_acceptance_based_Qlib_session'),('source_value','source_value')])
assert change['corrected_first_Qlib_session_after_later_date']=='2011-01-24'
result={'status':'one_qualitative_AMD_filing_date_exception_corrected_without_backfill','repair_plan_sha256':sha(card/'filing_date_repair_plan.json'),'original_result_sha256':plan['original_result_sha256'],'quarter_event_result_sha256':plan['quarter_event_result_sha256'],'selected_events':126,'unchanged_calendar_mappings':125,'changed_calendar_mappings':changes,'numeric_source_values_changed':0,'feature_values_changed':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'exact_first_public_timestamp_qualified':False}
(card/'filing_date_repair_result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
print(json.dumps({'status':result['status'],'changed_calendar_mappings':changes},ensure_ascii=False))

[4637:MainThread](2026-10-23 06:24:18,923) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[4637:MainThread](2026-10-23 06:24:19,309) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[4637:MainThread](2026-10-23 06:24:19,310) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{"status": "one_qualitative_AMD_filing_date_exception_corrected_without_backfill", "changed_calendar_mappings": [{"issuer": "AMD", "accession": "0001193125-11-011350", "raw_sha256": "9cf756c92cf2a5915106fb080451e7eea31ed3dace354279c000b62c19c0e106", "accepted_header": "20110120213715", "SEC_filing_date": "2011-01-21", "prior_acceptance_based_Qlib_session": "2011-01-21", "corrected_first_Qlib_session_after_later_date": "2011-01-24", "source_value": "qualitative_unknown"}]}


## 中文结论

126 个季度槽位中仅 AMD 2011-01-21 的定性未知事件发生日期修复：从 2011-01-21 延后到 2011-01-24；其余 125 个不变。仅调用一次 `D.calendar`，0 新请求、市场收益、拟合或账户。原始审计不覆盖写，完整修复见 `filing_date_repair_result.json`；首次公开时间仍未获证明。